In [ ]:
import pandas as pd
import forestplot as fp
import matplotlib.pyplot as plt
import matplotlib.collections as mcoll

from matplotlib.lines import Line2D

In [ ]:
df = pd.read_csv('source/csv/predictors_satisfaction.csv')
# df2 = pd.read_csv('source/csv/predictors_frequency_alignment.csv')
# df3 = pd.read_csv('source/csv/predictors_desire_alignment.csv')

In [ ]:
df.head(3)

In [ ]:
def get_stars(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return ''


df['Significance'] = df['p-value'].apply(get_stars)
# df2['Significance'] = df2['p-value'].apply(get_stars)
# df3['Significance'] = df3['p-value'].apply(get_stars)

In [ ]:
df['p-value-t'] = df['p-value'].apply(lambda x: f'{x:.3f}')
# df2['p-value-t'] = df2['p-value'].apply(lambda x: f'{x:.3f}')
# df3['p-value-t'] = df3['p-value'].apply(lambda x: f'{x:.3f}')

In [ ]:
def get_colors(p):
    if p < 0.05:
        return '#1f77b4'
    else:
        return '#bcbd22'


df['Color'] = df['p-value'].apply(get_colors)

In [ ]:
df = df.reset_index(drop=True)

In [ ]:
plot_df = df.copy()

In [ ]:
custom_order = [
    'Relationship Satisfaction',
    'Communication Quality',
    'Conflict Management',

    'Depressiveness',
    'Loneliness',
    'Self Esteem',
    'Health',
    'Life Satisfaction',

    'Extraversion',
    'Agreeableness',
    'Conscientiousness',
    'Openness',
    'Neuroticism',

    'Relationship Length',
    'Age',

    'Work Status',
    'Married',
    'Cohabitation',
    'Kids',
]

plot_df['Predictors'] = pd.Categorical(plot_df['Predictors'], categories=custom_order, ordered=True)
plot_df = plot_df.sort_values('Predictors').reset_index(drop=True)

In [ ]:
rename_dict0 = {
    'Age': 'Older age',  #1.015
    'Work Status': 'Have an occupation',  #1.144
    'Extraversion': 'Lower extraversion',  #0.980
    'Agreeableness': 'Lower agreeableness',  #0.976
    'Conscientiousness': 'Lower conscientiousness',  #0.989
    'Openness': 'Higher openness',  #1.003
    'Neuroticism': 'Lower neuroticism',  #0.994
    'Depressiveness': 'Lower depressiveness',  #0.996
    'Loneliness': 'Lower loneliness',  #0.878
    'Self Esteem': 'Higher self esteem',  #1.033
    'Life Satisfaction': 'Higher life satisfaction',  #1.004
    'Health': 'Better general health',  #1.076
    'Relationship Length': 'Shorter relationship length',  # 0.928
    'Married': 'Being single',  #0.856
    'Cohabitation': 'Live together',  #1.130
    'Kids': 'Have less kids',  #0.802
    'Communication Quality': 'Better communication',  #1.111
    'Relationship Satisfaction': 'Higher relationship satisfaction',  # 1.337
    'Conflict Management': 'Better conflict management',  #1.007
}

plot_df['Predictors'] = plot_df['Predictors'].replace(rename_dict0)

In [ ]:
# 2. Set up the plot aesthetics
plt.figure(figsize=(14, 8))
plt.style.use('default')

# 3. Create colors based on significance (p < 0.05)
colors = ['#1f77b4' if p < 0.05 else '#8d8d8d' for p in plot_df['p-value']]

for idx, (i, row) in enumerate(plot_df.iterrows()):
    plt.errorbar(
        x=row['AOR'],
        y=i,
        xerr=[[row['err_lower']], [row['err_upper']]],
        fmt='o',
        color=colors[idx],
        ecolor=colors[idx],
        capsize=4,
        markersize=7,
        linewidth=2
    )

plt.axvline(x=1.0, color='black', linestyle='--', alpha=0.7, zorder=0)

plt.yticks(range(len(plot_df)), plot_df['Predictors'], fontsize=16)
plt.xlabel(
    "Adjusted Odds Ratio (AOR)",
    x=0.385,
    ha='center',
    fontsize=14)

plt.title("Predictors of Satisfaction in touch", fontsize=22,
          x=0.385, loc='center', pad=15)

custom_lines = [Line2D([0], [0], color='#1f77b4', lw=3, marker='o'),
                Line2D([0], [0], color='#8d8d8d', lw=3, marker='o')]
# plt.legend(custom_lines, ['Significant (p < .05)', 'Not Significant'], loc='lower right', fontsize=16)
plt.legend(
    custom_lines,
    ['Significant (p < .05)', 'Not Significant'],
    loc='lower right',
    bbox_to_anchor=(1.0, 0.45),  # (x, y) position
    fontsize=16,
    frameon=False,
)
plt.gca().spines['top'].set_visible(True)
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['left'].set_visible(False)
plt.gca().spines['bottom'].set_visible(True)

plt.gca().invert_yaxis()

plt.savefig("output/img/paper/regression_stats.png",dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ax = fp.forestplot(
#     df,
#     estimate="AOR",
#     ll="Lower CI", hl="Upper CI",
#     varlabel="Predictors",
#     xlabel="Adjusted Odds Ratio",
#     groupvar="Category",
#     decimal_precision=2,
#     color_alt_rows=False,
#     annote=["est_ci", "p-value-t", "Significance"],
#     annoteheaders=["AOR (95% CI)", "p-value", "Sig.    "],
#     # extra space in the last header to add some padding to the plot
#     figsize=(8, 8),
#     table=True,
#     **{
#         'fontfamily': 'DejaVu Sans Mono',
#         'ylabel1_size': 12,
#
#         'marker': 'o',
#         'markersize': 100,
#         'markercolor': '#1f77b4',
#
#         'linecolor': '#1f77b4',
#         'linestyle': '-',
#     },
#     xlim=(0.7, 1.5),
#
# )
#
# for collection in ax.collections:
#     print(collection)
#     if isinstance(collection, mcoll.PathCollection):
#         collection.set_zorder(10)  # A high number forces it to the top layer
#         collection.set_alpha(1.0)
#     elif isinstance(collection, mcoll.LineCollection):
#         collection.set_zorder(10)  # A high number forces it to the top layer
#         collection.set_alpha(1.0)
#
# ax.axvline(x=1, color='grey', linestyle='--', linewidth=1.5)
#
# plt.savefig("output/img/alignment/alignment_forest.png", dpi=300, bbox_inches='tight')
# plt.show()

In [ ]:
# ax2 = fp.forestplot(
#     df2,
#     estimate="AOR",
#     ll="Lower CI", hl="Upper CI",
#     varlabel="Predictors",
#     # ylabel="Est.(95% Conf. Int.)",
#     xlabel="Adjusted Odds Ratio",
#     groupvar="Category",
#     # pval="p-value",
#     decimal_precision=2,
#     color_alt_rows=False,
#     annote=["est_ci", "p-value-t", "Significance"],
#     annoteheaders=["AOR (95% CI)", "p-value", "Sig.    "],
#     figsize=(6, 4),
#     table=True,
#
#     **{
#         'fontfamily': 'DejaVu Sans Mono',
#         'ylabel1_size': 12,
#
#         'marker': 'o',
#         'markersize': 100,
#         'markercolor': '#1f77b4',
#
#         'linecolor': '#1f77b4',
#         'linestyle': '-',
#     },
#     xlim=(0.85, 1.41),
# )
#
# for collection in ax2.collections:
#     print(collection)
#     if isinstance(collection, mcoll.PathCollection):
#         collection.set_zorder(10)  # A high number forces it to the top layer
#         collection.set_alpha(1.0)
#     elif isinstance(collection, mcoll.LineCollection):
#         collection.set_zorder(10)  # A high number forces it to the top layer
#         collection.set_alpha(1.0)
# ax2.axvline(x=1, color='grey', linestyle='--', linewidth=1.5)
#
# plt.savefig("output/img/alignment/alignment_forest_frequency.png", dpi=300, bbox_inches='tight')
#
# plt.show()

In [ ]:
# ax3 = fp.forestplot(
#    df3,
#     estimate="AOR",
#     ll="Lower CI", hl="Upper CI",
#     varlabel="Predictors",
#     # ylabel="Est.(95% Conf. Int.)",
#     xlabel="Adjusted Odds Ratio",
#     groupvar="Category",
#     # pval="p-value",
#     decimal_precision=2,
#     color_alt_rows=False,
#     annote=["est_ci", "p-value-t", "Significance"],
#     annoteheaders=["AOR (95% CI)", "p-value", "Sig.    "],
#     figsize=(6, 4),
#     table=True,
#
#     **{
#         'fontfamily': 'DejaVu Sans Mono',
#         'ylabel1_size': 12,
#
#         'marker': 'o',
#         'markersize': 100,
#         'markercolor': '#1f77b4',
#
#         'linecolor': '#1f77b4',
#         'linestyle': '-',
#     },
#     xlim=(0.85, 1.11),
# )
#
# for collection in ax3.collections:
#     print(collection)
#     if isinstance(collection, mcoll.PathCollection):
#         collection.set_zorder(10)  # A high number forces it to the top layer
#         collection.set_alpha(1.0)
#     elif isinstance(collection, mcoll.LineCollection):
#         collection.set_zorder(10)  # A high number forces it to the top layer
#         collection.set_alpha(1.0)
# ax3.axvline(x=1, color='grey', linestyle='--', linewidth=1.5)
#
# plt.savefig("output/img/alignment/alignment_forest_desire.png",dpi=300, bbox_inches='tight')
#
# plt.show()